# Task 5 — Historical trend forecasting

This executed notebook reads frozen experimental outputs. Annual release-year cohorts are the assignment focus; monthly date-added cohorts are a distinct supplementary target. Neither is a complete Netflix release pipeline. Projections are relative to completed historical periods, not the current calendar date. The snapshot ends September 25, 2021; 2021 release cohorts and September additions are excluded from modeling.

Selection uses expanding chronological CV before the final test. There are no shuffled splits or future observed lag values. Final projections refit the selected model on all completed periods after evaluation. Test results remain frozen and must not be used to change the selected model.


In [1]:
from netflix_ml.tasks.task_05.service import load_forecasting

service = load_forecasting()
result = service.results
print('Run:', result['run_id'])
print('Snapshot:', result['snapshot_date'])
print('Entities:', result['audit']['entity_rows'])
print('Date anomalies retained:', len(result['audit']['date_anomalies']))
print('Excluded 2021 release-year cohort:', result['audit']['annual_excluded_2021'])
print('Excluded September additions:', result['audit']['monthly_excluded_september_2021'])


Run: 20261106T122211417836Z
Snapshot: 2021-09-25
Entities: 8783
Date anomalies retained: 14
Excluded 2021 release-year cohort: 592
Excluded September additions: 183


In [2]:
import pandas as pd

for row in result['series']:
    print(row['target'], row['group'], 'selected', row['model'])
    print('Test metrics:', row['test_metrics'])
    print(pd.DataFrame([{'model':c['model'],'cv_mae':c['cv_mae'],'test_mae':c['test_metrics']['mae']} for c in row['comparison']]).to_string(index=False))


releases all selected ridge_1
Test metrics: {'mae': 2395.3692217918656, 'rmse': 3074.4155086000274, 'smape': 89.07462540502266, 'wape': 2.306011284516838, 'mase': 42.966264068015526}
         model     cv_mae    test_mae
    last_value 145.600000  137.750000
        mean_3 181.000000  436.416667
         drift 122.573911  101.750000
       ridge_1 109.226869 2395.369222
      ridge_10 123.942367  719.313438
     ridge_100 209.755762  377.846915
forest_shallow 192.549944  447.869365
   forest_full 192.549944  447.869365
releases movies selected ridge_1
Test metrics: {'mae': 1801.7014378784452, 'rmse': 2319.4918265729334, 'smape': 95.16143177937018, 'wape': 2.689106623699172, 'mase': 44.011027490160494}
         model     cv_mae    test_mae
    last_value 101.500000   95.500000
        mean_3 126.900000  231.000000
         drift  84.956114  134.562500
       ridge_1  78.959675 1801.701438
      ridge_10  89.017155  567.470814
     ridge_100 148.600396  197.269975
forest_shallow 134.7733

In [3]:
for row in result['series']:
    print(row['target'],row['group'],'origin',row['forecast_origin'])
    print(pd.DataFrame(row['forecast']).to_string(index=False))
    print('Empirical CV band errors:',row['band_residual_count'],'test coverage:',row['test_band_coverage'])


releases all origin 2020-01-01
    period   predicted      lower       upper
2021-01-01  938.733149 738.392208 1139.074090
2022-01-01  969.166125 768.825184 1169.507066
2023-01-01 1032.179839 831.838898 1232.520780
Empirical CV band errors: 10 test coverage: 0.0
releases movies origin 2020-01-01
    period  predicted      lower      upper
2021-01-01 477.225899 344.822091 609.629708
2022-01-01 503.394570 370.990762 635.798378
2023-01-01 578.587732 446.183924 710.991540
Empirical CV band errors: 10 test coverage: 0.0
releases tv origin 2020-01-01
    period  predicted      lower      upper
2021-01-01 520.045498 450.784071 589.306925
2022-01-01 585.078409 515.816982 654.339837
2023-01-01 659.757085 590.495658 729.018512
Empirical CV band errors: 10 test coverage: 0.0
additions all origin 2021-08-01
    period  predicted      lower      upper
2021-09-01 182.911214 150.207671 215.614756
2021-10-01 187.622768 154.919225 220.326311
2021-11-01 174.339850 141.636307 207.043392
2021-12-01 179.83

## Interpretation and limitations

Compare each CV-selected model with last-value, drift/mean and (monthly) seasonal-naive baselines on the untouched test. A worse test result is a failure of generalization, not a reason to reselect from test scores. Annual cohorts show rapid growth followed by flattening; recursive extrapolation can amplify errors across that change. Sparse annual history and overlapping validation origins limit evidence.

The pooled 80th-percentile absolute CV residual forms an exploratory error band. It is not a calibrated interval; annual validation is two steps whereas test/projection are longer. Coverage on the independent test must be disclosed. Snapshot survivorship and retrospective backfills mean these are not true point-in-time historical archives. Separate content-type models are not reconciled and their forecasts need not sum.


![releases historical as-of projection](../../reports/task-05/figures/releases-all.png)

![additions historical as-of projection](../../reports/task-05/figures/additions-all.png)